# Election Program Parser

#### This script is useful for election program pdf's

In [ ]:
import os
import re
import csv
import fitz  # PyMuPDF
import pytesseract
import pandas as pd
from datetime import datetime
from PIL import Image
import PyPDF2
from rapidfuzz import process, fuzz
from IPython.display import display, HTML
import numpy as np
import unicodedata
from urllib.parse import unquote
from pathlib import Path
from typing import Optional, Dict, Any, Tuple

folder_path = r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Corporate blogs" #set location of file 
output_csv = "corp_blogs_parsed.csv" #set name of output file

In [3]:
# Check if folder is accessible
print("Exists:", os.path.exists(folder_path))
print("Is Directory:", os.path.isdir(folder_path))

# List first few items in the top-level folder
try:
    files = os.listdir(folder_path)
    print(f"Total items detected in top-level: {len(files)}")
    print("First 10 items:", files[:10])
except Exception as e:
    print(f"Error listing files: {e}")

# Collect PDFs from all subfolders
pdf_files = []
for root, dirs, files in os.walk(folder_path):
    for f in files:
        if f.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, f))

print(f"Total PDFs found (recursive): {len(pdf_files)}")
print("First 10 PDFs:", pdf_files[:10])

Exists: True
Is Directory: True
Total items detected in top-level: 24
First 10 items: ['ABN AMRO', 'Accenture', 'Achmea', 'AEGON', 'Alliander', 'Bol.com', 'Booking.com', 'Bosch', 'Capgemini', 'Deloitte']
Total PDFs found (recursive): 79
First 10 PDFs: ['C:\\Users\\joly-\\OneDrive - Universiteit Utrecht\\UU\\Dataschool\\HUMAN\\Data\\Corporate blogs\\ABN AMRO\\ABN AMRO biedt bedrijven complete bescherming tegen cybercrime _ ABN AMRO.pdf', 'C:\\Users\\joly-\\OneDrive - Universiteit Utrecht\\UU\\Dataschool\\HUMAN\\Data\\Corporate blogs\\ABN AMRO\\Kunstmatige intelligentie_ Vriend of vijand_ _ ABN AMRO.pdf', 'C:\\Users\\joly-\\OneDrive - Universiteit Utrecht\\UU\\Dataschool\\HUMAN\\Data\\Corporate blogs\\ABN AMRO\\Samen sterk tegen cyberaanvallen van morgen _ ABN AMRO.pdf', 'C:\\Users\\joly-\\OneDrive - Universiteit Utrecht\\UU\\Dataschool\\HUMAN\\Data\\Corporate blogs\\ABN AMRO\\Van digitale assistent tot virtuele BFF _ ABN AMRO.pdf', 'C:\\Users\\joly-\\OneDrive - Universiteit Utrecht\\UU\

In [4]:
def extract_text_from_pdf(pdf_path, max_ocr_pages=1):
    """Extract text using PyMuPDF; if a page fails, OCR that page."""
    text_parts = []
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            try:
                t = doc[page_num].get_text("text")  # block-aware plain text
            except Exception:
                t = ""
            if not t.strip():
                # Fallback OCR (only first N pages to keep it fast)
                if page_num < max_ocr_pages:
                    try:
                        page = doc.load_page(page_num)
                        pix = page.get_pixmap()
                        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                        import pytesseract
                        t = pytesseract.image_to_string(img, lang="nld+eng")
                    except Exception:
                        t = ""
            text_parts.append(t)
    return "\n".join(text_parts)

In [5]:


# --- Normalisation helper ---
def norm(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("+", " plus ")
    s = re.sub(r"[_\-/.]", " ", s)        # separators -> spaces
    s = s.replace("’", "'").replace("'", "")
    s = s.replace(".", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# --- PDF text extraction ---
def extract_text_from_pdf(pdf_path, max_ocr_pages=1):
    """Extract text using PyMuPDF; if a page fails, OCR that page."""
    text_parts = []
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            try:
                t = doc[page_num].get_text("text")  # block-aware plain text
            except Exception:
                t = ""
            if not t.strip():
                # Fallback OCR (only first N pages to keep it fast)
                if page_num < max_ocr_pages:
                    try:
                        page = doc.load_page(page_num)
                        pix = page.get_pixmap()
                        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                        import pytesseract
                        t = pytesseract.image_to_string(img, lang="nld+eng")
                    except Exception:
                        t = ""
            text_parts.append(t)
    return "\n".join(text_parts)

def build_pdf_dataframe(root_folder: str, max_ocr_pages=1) -> pd.DataFrame:
    records = []
    for dirpath, _, files in os.walk(root_folder):
        for f in files:
            if f.lower().endswith(".pdf"):
                full_path = os.path.join(dirpath, f)
                parts = os.path.normpath(full_path).split(os.sep)
                subfolder = norm(parts[-2]) if len(parts) >= 2 else None
                top_folder = norm(parts[0]) if parts else None

                # Extract text once
                text = extract_text_from_pdf(full_path, max_ocr_pages=max_ocr_pages)

               

                records.append({
                    "subfolder": subfolder,
                    "top_folder": top_folder,
                    "filename": f,
                    "full_path": full_path,
                    "text": text
                })

    return pd.DataFrame(records)

In [ ]:
root_folder = folder_path   # path to your root folder
df = build_pdf_dataframe(root_folder)

In [7]:
df

,subfolder,top_folder,filename,full_path,text
0,abn amro,c:,ABN AMRO biedt bedrijven complete bescherming ...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,ABN AMRO biedt midden- en grootbedrijf complet...
1,abn amro,c:,Kunstmatige intelligentie_ Vriend of vijand_ _...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Kunstmatige intelligentie: Vriend of vijand?\n...
2,abn amro,c:,Samen sterk tegen cyberaanvallen van morgen _ ...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Samen sterk tegen cyberaanvallen van morgen\nN...
3,abn amro,c:,Van digitale assistent tot virtuele BFF _ ABN ...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Van digitale assistent tot virtuele BFF\nBlog\...
4,accenture,c:,Beyond the Hype_ Why Agentic AI is Closer Than...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,BLOG\nBeyond the hype:\nWhy agentic AI is\nclo...
...,...,...,...,...,...
74,tno,c:,How TNO is leading the drive towards sovereign...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
75,tno,c:,Large dataset news organizations for Dutch AI ...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
76,tno,c:,New AI Lab for effective and responsible overs...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
77,ziggo,c:,From Gut Feeling to Data-Driven Decisions_ How...,C:\Users\joly-\OneDrive - Universiteit Utrecht...,From Gut Feeling to Data-Driven\nFrom Gut Feel...


In [8]:
df = df.drop(columns=["top_folder", 'full_path'])  # drop full_path for privacy

In [9]:
test_pdf = os.path.join(folder_path, pdf_files[0])  # Pick first PDF
print(f"Testing file: {test_pdf}")

try:
    text = extract_text_from_pdf(test_pdf)  # Run extraction
    print("Extracted text (first 500 characters):")
    print(text[:500])  # Show first 500 characters of extracted text
except Exception as e:
    print(f"Error extracting text from {test_pdf}: {e}")

Testing file: C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Corporate blogs\ABN AMRO\ABN AMRO biedt bedrijven complete bescherming tegen cybercrime _ ABN AMRO.pdf
Extracted text (first 500 characters):
ABN AMRO biedt midden- en grootbedrijf complete service tegen
cybercriminaliteit
Persbericht
Financiële criminaliteit
1 februari 2023, 09:00
Eunice Koekkoek
Senior Persvoorlichter
Het aantal bedrijven dat te maken heeft met cybercriminaliteit
neemt sterk toe. Dit kost het Nederlandse bedrijfsleven miljarden
euro’s per jaar. Als eerste bank in Nederland biedt ABN AMRO in
samenwerking met cybersecurity leverancier MMOX nu met
Cyber Veilig & Zeker een complete service tegen cyberrisico’s in
de vorm


In [10]:
df = df.rename(columns={"subfolder": "company", "filename": "blog_title", "text": "blog_content"})
df

,company,blog_title,blog_content
0,abn amro,ABN AMRO biedt bedrijven complete bescherming ...,ABN AMRO biedt midden- en grootbedrijf complet...
1,abn amro,Kunstmatige intelligentie_ Vriend of vijand_ _...,Kunstmatige intelligentie: Vriend of vijand?\n...
2,abn amro,Samen sterk tegen cyberaanvallen van morgen _ ...,Samen sterk tegen cyberaanvallen van morgen\nN...
3,abn amro,Van digitale assistent tot virtuele BFF _ ABN ...,Van digitale assistent tot virtuele BFF\nBlog\...
4,accenture,Beyond the Hype_ Why Agentic AI is Closer Than...,BLOG\nBeyond the hype:\nWhy agentic AI is\nclo...
...,...,...,...
74,tno,How TNO is leading the drive towards sovereign...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
75,tno,Large dataset news organizations for Dutch AI ...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
76,tno,New AI Lab for effective and responsible overs...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
77,ziggo,From Gut Feeling to Data-Driven Decisions_ How...,From Gut Feeling to Data-Driven\nFrom Gut Feel...


In [21]:
df.to_csv('corp_blogs_parsed.csv')